# 10 — The Sinkhorn Algorithm

Bend the Gibbs kernel onto the marginal constraints by nothing more than rescaling rows and columns in turn. Five lines, two matrix products per step — the algorithm that made optimal transport fast.

**Prerequisites:** `09_entropic_regularization`.

**What you'll be able to do**
- Run the five-line Sinkhorn iteration and confirm the resulting plan's marginals.
- See its geometric (exponential) convergence.
- Read the $\varepsilon$ trade-off: sharp near-LP plans at small $\varepsilon$, blurry near-independent plans at large $\varepsilon$.

The Gibbs kernel from the last notebook is not yet a transport plan — its rows and columns do not sum to $a$ and $b$. Sinkhorn's idea is that you can *bend* the kernel onto those marginals by alternately rescaling its rows and its columns. Five lines of code, two matrix–vector products per step, geometric convergence — and it revolutionised computational optimal transport.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ot

from qot_course import viz
from qot_course.transport.discrete import squared_euclidean_cost, transport_cost
from qot_course.transport.sinkhorn import sinkhorn, sinkhorn_iterations_log

np.random.seed(42)
viz.use_course_style()

## 1. Five lines of matrix scaling

The optimality conditions of the entropic problem give $P^\star_\varepsilon = \mathrm{diag}(u)\, K\, \mathrm{diag}(v)$, with $u, v$ fixed by the marginal constraints $\mathrm{diag}(u)(Kv) = a$ and $\mathrm{diag}(v)(K^\top u) = b$. Sinkhorn alternates them:

```python
K = exp(-C / epsilon)
u = ones(n); v = ones(m)
while not converged:
    v = b / (K.T @ u)
    u = a / (K @ v)
```

That is the entire algorithm. Each step costs two matrix–vector products — $O(nm)$ — against $O(n^3)$ for the LP, and it converges geometrically (Franklin & Lorenz, 1989). Let's run it and check the plan it returns.

In [ ]:
# A small problem: 4 source atoms, 5 target atoms, squared ground cost.
rng = np.random.default_rng(0)
a = rng.random(4); a = a / a.sum()
b = rng.random(5); b = b / b.sum()
cost = squared_euclidean_cost(np.arange(4, dtype=float), np.linspace(0, 3, 5))

epsilon = 0.5
plan_sk = sinkhorn(a, b, cost, epsilon=epsilon, n_iter=2000, tol=1e-12)

print(f"row sums  P 1   = {np.round(plan_sk.sum(axis=1), 6).tolist()}")
print(f"target    a     = {np.round(a, 6).tolist()}")
print(f"col sums  P^T 1 = {np.round(plan_sk.sum(axis=0), 6).tolist()}")
print(f"target    b     = {np.round(b, 6).tolist()}")

lp_cost = float(ot.emd2(a, b, cost))
sk_cost = transport_cost(plan_sk, cost)
pot_plan = ot.sinkhorn(a, b, cost, epsilon, numItermax=2000, stopThr=1e-12)
print(f"\nLP cost        <C, P*_0>   = {lp_cost:.4f}")
print(f"Sinkhorn cost  <C, P*_eps> = {sk_cost:.4f}   (eps = {epsilon})")
print(f"agreement with POT sinkhorn: {bool(np.allclose(plan_sk, pot_plan, atol=1e-4))}")

**Read the output.** The plan's marginals match $a$ and $b$ to floating-point precision — Sinkhorn returns a feasible coupling. Its transport cost is *strictly larger* than the LP optimum: the entropy bonus biases the plan toward spread, and that bias shrinks as $\varepsilon \to 0$. And our five-line routine agrees with POT's reference solver.

## 2. Geometric convergence

How fast does the iteration close the marginal constraints? We track the row-marginal error $\|P\mathbf{1} - a\|_\infty$ across iterations.

In [ ]:
err_row, _ = sinkhorn_iterations_log(a, b, cost, epsilon=epsilon, n_iter=200)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.semilogy(err_row, color=viz.SOURCE_COLOR, lw=2)
ax.axhline(1e-9, color=viz.TARGET_COLOR, linestyle="--", alpha=0.7, label="~1e-9 floor")
ax.set_xlabel("Sinkhorn iteration")
ax.set_ylabel(r"row-marginal error  $\|P\mathbf{1} - a\|_\infty$  (log scale)")
ax.set_title(f"Geometric convergence of Sinkhorn  (eps = {epsilon})", pad=12)
ax.legend()
plt.show()

**Read the figure.** A straight line on a log scale is **geometric (exponential)** convergence: each iteration shrinks the marginal error by a constant factor. At $\varepsilon = 0.5$ we hit machine precision in about $200$ iterations on this small problem. The rate depends on $\varepsilon$ — smaller $\varepsilon$ converges more slowly and can underflow, which is when practitioners switch to a log-domain implementation.

## 3. The $\varepsilon$ trade-off — sharp versus blurry

The entropy term blurs the plan. Tiny $\varepsilon$ keeps it sparse and close to the LP; large $\varepsilon$ pushes it toward the independent coupling. Here is the same problem solved at four values of $\varepsilon$.

In [ ]:
positions = np.linspace(0.0, 9.0, 16)

def bump(center, sigma=1.2):
    p = np.exp(-0.5 * ((positions - center) / sigma) ** 2)
    return p / p.sum()

src = bump(2.0)
tgt = bump(7.0)
C_demo = squared_euclidean_cost(positions, positions)

epsilons = [0.05, 0.5, 5.0, 50.0]
plans = [sinkhorn(src, tgt, C_demo, epsilon=eps, n_iter=2000, tol=1e-10) for eps in epsilons]

fig, axes = plt.subplots(1, 4, figsize=(13, 3.6))
for ax, plan, eps in zip(axes, plans, epsilons):
    im = ax.imshow(plan, cmap=viz.CMAP_PLAN, origin="lower", aspect="equal")
    ax.set_title(rf"$\varepsilon = {eps}$", pad=8)
    ax.set_xlabel("target position")
    ax.grid(False)
    if ax is axes[0]:
        ax.set_ylabel("source position")
    fig.colorbar(im, ax=ax, shrink=0.85)
fig.suptitle(r"Entropic plans across $\varepsilon$: sharp diagonal $\to$ blurry independent coupling",
             fontsize=13, fontweight="bold", y=1.04)
plt.tight_layout()
plt.show()

**Read the four heatmaps.** At $\varepsilon = 0.05$ the plan is a sharp diagonal stripe — essentially the LP solution, moving mass the shortest distance. At $\varepsilon = 0.5$ the stripe broadens. At $\varepsilon = 5$ the diagonal is mostly gone, replaced by a smooth bilinear pattern $\approx u_i v_j$. At $\varepsilon = 50$ the plan is nearly the independent coupling $\mathrm{src} \otimes \mathrm{tgt}$ — the entropy bonus has swamped the transport cost. Choosing $\varepsilon$ is the familiar regularisation trade-off: small enough to keep the geometry, large enough to stay fast and stable.

## Your turn

1. **The $\varepsilon$ bias.** For the $4 \times 5$ problem, plot the Sinkhorn cost against $\varepsilon \in [0.05, 5]$ and compare to the LP cost. At what $\varepsilon$ does the relative bias drop below $1\%$? *(The bias scales like $\varepsilon \log(nm)$ asymptotically.)*
2. **Log-domain Sinkhorn.** At $\varepsilon = 0.01$ on the bumps example, $K = e^{-C/\varepsilon}$ underflows below $10^{-300}$. Rewrite the iteration in log-space (replace the matrix–vector products by `logsumexp`) and reach $\varepsilon = 0.01$ without `nan`s.
3. **Debiased Sinkhorn divergence.** Compute $S_\varepsilon(\mu, \nu) = \mathrm{OT}_\varepsilon(\mu, \nu) - \tfrac{1}{2}\mathrm{OT}_\varepsilon(\mu, \mu) - \tfrac{1}{2}\mathrm{OT}_\varepsilon(\nu, \nu)$ (Feydy et al., 2019) on the bumps and confirm $S_\varepsilon(\mu, \mu) = 0$ while $S_\varepsilon(\mu, \nu) > 0$.

## What you built

- You ran the five-line Sinkhorn iteration and confirmed the plan's marginals match $a$ and $b$.
- You saw geometric convergence — a straight line on a log scale.
- You read the $\varepsilon$ trade-off, from a sharp near-LP diagonal to a blurry near-independent plan.

You now have a fast, scalable optimal-transport solver, and you understand the one knob it asks you to set. The next notebook reveals *why* this iteration works — and ties it back to the information geometry of Module 2.

## What's next

In `11_amari_bridge` we close the arc with the identity that makes Sinkhorn more than a trick: the entropic plan is the **KL projection** of the Gibbs kernel onto the transportation polytope. Wasserstein meets KL — and the M2↔M3 loop closes.

## References

- M. Cuturi, "Sinkhorn distances: lightspeed computation of optimal transport", *NeurIPS* (2013).
- R. Sinkhorn, "A relationship between arbitrary positive matrices and doubly stochastic matrices", *Ann. Math. Statist.* 35, 876–879 (1964).
- J. Feydy et al., "Interpolating between optimal transport and MMD using Sinkhorn divergences", *AISTATS* (2019).
- G. Peyré & M. Cuturi, *Computational Optimal Transport*, NoW (2019), chs. 4–5. DOI:10.1561/2200000073

**Previous:** `notebooks/03_ClassicalOptimalTransport/09_entropic_regularization.ipynb` · **Next:** `notebooks/03_ClassicalOptimalTransport/11_amari_bridge.ipynb`